### Imports

In [7]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import sys
from pathlib import Path

ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "paths.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate project root containing src/paths.py")
sys.path.insert(0, str(ROOT))


from src.paths import *
from src.graph_utils import (
    build_grid_graph,
    calculate_graph_metrics,
    add_solar_distance
)

### Calculate centrality

In [8]:
substations_path = GIS / "grid_substations.csv"
lines_path = GIS / "transmission_lines.csv"
solar_path = GIS / "solar_power_plants.csv"
if all(path.is_file() for path in (substations_path, lines_path, solar_path)):
    substations = pd.read_csv(substations_path)
    lines = pd.read_csv(lines_path)
    solar_plants = pd.read_csv(solar_path)
else:
    missing = [path.name for path in (substations_path, lines_path, solar_path) if not path.is_file()]
    print(f"Analysis skipped because GIS input files are missing from {GIS}: {', '.join(missing)}")
    print("Fill the corresponding templates under datasets/templates/ and place completed CSVs in the GIS folder.")
    substations = pd.DataFrame(columns=["bus_id", "voltage_kv", "latitude", "longitude"])
    lines = pd.DataFrame(columns=["from_bus", "to_bus", "length_km"])
    solar_plants = pd.DataFrame(columns=["plant_id", "latitude", "longitude"])

substations["bus_id"] = substations["bus_id"].astype(str)
lines["from_bus"] = lines["from_bus"].astype(str)
lines["to_bus"] = lines["to_bus"].astype(str)
if "bus_id" in solar_plants.columns:
    solar_plants["bus_id"] = solar_plants["bus_id"].astype(str)

G = build_grid_graph(substations, lines)
metrics = calculate_graph_metrics(G)
display(metrics.head(10))

Analysis skipped because GIS input files are missing from D:\My_Project\Analyse\4G-Network-Coverage-Optimization\datasets\raw\gis: grid_substations.csv, transmission_lines.csv, solar_power_plants.csv
Fill the corresponding templates under datasets/templates/ and place completed CSVs in the GIS folder.


,bus_id,degree,degree_centrality,betweenness_centrality,closeness_centrality


### Calculate distance to solar plants

In [9]:
metrics = add_solar_distance(
    G,
    metrics,
    solar_plants
)

metrics = metrics.merge(
    substations[["bus_id", "voltage_kv"]],
    on="bus_id",
    how="left"
)

display(metrics.head())

,bus_id,degree,degree_centrality,betweenness_centrality,closeness_centrality,solar_distance_km,voltage_kv


### Visualize betweenness centrality

In [10]:
if metrics.empty or metrics["betweenness_centrality"].dropna().empty:
    print("Centrality plot skipped: no grid metrics are available. Load completed GIS inputs and rerun the analysis.")
else:
    ax = metrics.sort_values("betweenness_centrality", ascending=False).head(20).plot(
        x="bus_id", y="betweenness_centrality", kind="bar",
        legend=False, figsize=(12, 6)
    )
    ax.set_title("Top 20 Substations by Betweenness Centrality")
    ax.set_ylabel("Betweenness Centrality")
    ax.set_xlabel("Substation")
    ax.figure.tight_layout()
    ax.figure.savefig(GRAPHS / "betweenness_centrality.png", dpi=300)
    plt.show()

Centrality plot skipped: no grid metrics are available. Load completed GIS inputs and rerun the analysis.


### Save Graph Theory results

In [11]:
metrics.to_csv(
    PROCESSED / "graph_theory_metrics.csv",
    index=False
)